In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import spectrogram
import librosa
import librosa.display
from scipy import signal

def Probability(Data, M, tau):
    z = np.zeros((len(Data) - (M - 1) * tau, M))

    for i in range(len(Data) - (M - 1) * tau):
        for j in range(M):
            z[i][j] = Data[i + j * tau] 

    e = np.argsort(z, axis = 1)
        
    pattern_count = {}
    for pattern in e:
        key = tuple(pattern)
        pattern_count[key] = pattern_count.get(key, 0) + 1
            
    total = sum(pattern_count.values())
    P = np.array([count / total for count in pattern_count.values()])
    return P

def Probability_U(P):
    return np.full(shape=len(P), fill_value=1/len(P))

def Probability_Const(P):
    p = np.zeros(len(P))
    p[0] = 1
    return p

def Entropy(P):
    P = P[P > 0]
    return -np.sum(P * np.log2(P))

def J(P1, P2):
    P3 = np.add(P1, P2) / 2
    return Entropy(P3) - 0.5 * (Entropy(P1) + Entropy(P2))

def Entropy_Complexity(data, M = 3, tau = 1):
    prob = Probability(data, M, tau)
    prob_u = Probability_U(prob)
    prob_const = Probability_Const(prob)
    entropy = Entropy(prob) / Entropy(prob_u)
    complexity = entropy * (J(prob, prob_u) / J(prob_u, prob_const))
    return entropy, complexity

def Hurst(data):
    lags = range(2, min(100, len(data)//2))
    tau = [np.sqrt(np.std(np.subtract(data[lag:], data[:-lag]))) for lag in lags]
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return poly[0]

def Fract(data):
    L = []
    x = np.array(data)
    N = len(x)
    for k in range(1, 11):
        Lk = []
        for m in range(k):
            Lmk = 0
            for i in range(1, (N - m) // k):
                Lmk += abs(x[m + i * k] - x[m + (i - 1) * k])
            Lmk = Lmk * (N - 1) / ((N - m) / k) / k
            Lk.append(Lmk)
        L.append(np.mean(Lk))
    L = np.array(L)
    fract = -np.polyfit(np.log(np.arange(1, 11)), np.log(L), 1)[0]
    return fract


In [ ]:
DataType = []
DataEntropy = []
DataComplexity = []
DataHurst = []
DataFract = []

for i in range(408):
    s0 = str(i + 1)
    while len(s0) < 4:
        s0 = '0' + s0
    s1 = 'Test/a' + s0 + '.wav'
    s2 = 'Test/a' + s0 + '.hea'

    f = open(s2)
    s = [x for x in f]
    s = s[-1][2:-1]
    DataType.append(s)

    data, fs = librosa.load(s1, sr = None)
    ent, com = Entropy_Complexity(data)
    hur = Hurst(data)
    fr = Fract(data)
    DataEntropy.append(ent)
    DataComplexity.append(com)
    DataHurst.append(hur)
    DataFract.append(fr)


In [ ]:
cnt1 = 0
cnt2 = 0
cnt3 = 0
cnt4 = 0
for i in range(407, -1, -1):
    cnt = 0
    if (DataEntropy[i] >= 0.764):
        cnt += 1
    
    if (DataComplexity[i] <= 0.168):
        cnt += 1

    if (DataHurst[i] >= 0.089):
        cnt += 1

    if (DataFract[i] >= 0.151):
        cnt += 1
    
    if (cnt < 2):
        if (DataType[i] == 'Normal'):
            cnt1 += 1
        else:
            cnt2 += 1
    else:
        if (DataType[i] == 'Abnormal'):
            cnt3 += 1
        else:
            cnt4 += 1
print("Количество правильно определенных Normal точек:", cnt1, str(cnt1 * 100 / DataType.count('Normal')) + '%')
print("Количество определенных Abnormal точек, как Normal:", cnt2, str(cnt2 * 100 / DataType.count('Abnormal')) + '%')
print("Количество правильно определенных Abnormal точек:", cnt3, str(cnt3 * 100 / DataType.count('Abnormal')) + '%')
print("Количество определенных Normal точек, как Abnormal:", cnt4, str(cnt4 * 100 / DataType.count('Normal')) + '%')

Количество правильно определенных Normal точек: 130 83.87096774193549%
Количество определенных Abnormal точек, как Normal: 65 25.691699604743082%
Количество правильно определенных Abnormal точек: 188 74.30830039525692%
Количество определенных Normal точек, как Abnormal: 25 16.129032258064516%
